# Create filtered Datasets by max area threshold

#### <b>Objective</b>: Create two datasets, which only have annotation with an box area lower than the max fence detected of 2.5% and 5%

#### 0 - Create Folders

In [1]:
import os

def create_folder_dataset(directory, subfolder, labels):

    os.mkdir(directory)

    for sb in subfolder:
        os.mkdir(directory + "/" + sb)
        for sb1 in ['images', 'labels']:
            os.mkdir(directory + "/" + sb + "/" + sb1)

            for sb2 in ['train', 'test']:
                os.mkdir(directory + "/" + sb + "/" + sb1 + "/" + sb2)


                if "3Classes" in sb:
                    os.mkdir(directory + "/" + sb + "/" + sb1 + "/" + sb2 + "/" + "BlackRot")
                    os.mkdir(directory + "/" + sb + "/" + sb1 + "/" + sb2 + "/" + "Esca")
                    os.mkdir(directory + "/" + sb + "/" + sb1 + "/" + sb2 + "/" + "LeafBlight")

                elif "BlackRot" in sb:
                    os.mkdir(directory + "/" + sb + "/" + sb1 + "/" + sb2 + "/" + "BlackRot")

                elif "Esca" in sb:
                    os.mkdir(directory + "/" + sb + "/" + sb1 + "/" + sb2 + "/" + "Esca")

                elif "LeafBlight" in sb:
                    os.mkdir(directory + "/" + sb + "/" + sb1 + "/" + sb2 + "/" + "LeafBlight")



labels = ["BlackRot", "Esca", "LeafBlight"]


create_folder_dataset("DatasetOrganized/WithoutOutliers_1_20", 
                    ["DatasetBlackRot_WithoutOutliers_1_20", 
                       "DatasetEsca_WithoutOutliers_1_20",
                        "DatasetLeafBlight_WithoutOutliers_1_20",
                        "Dataset3Classes_WithoutOutliers_1_20"   
                    ],
                    labels)

create_folder_dataset("DatasetOrganized/WithoutOutliers_1_50", 
                    ["DatasetBlackRot_WithoutOutliers_1_50", 
                       "DatasetEsca_WithoutOutliers_1_50",
                        "DatasetLeafBlight_WithoutOutliers_1_50" ,
                        "Dataset3Classes_WithoutOutliers_1_50"   
                    ],
                    labels)

#### 0.5 - Visualization of data function

In [2]:
import cv2
import matplotlib.pyplot as plt
import os
from PIL import Image
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd

def verify_number_annot(path1, path2, path3, labels):
    annot_path1 = pd.DataFrame([], columns = ["x", "y", "w", "h"])
    annot_path2 = pd.DataFrame([], columns = ["x", "y", "w", "h"])
    annot_path3 = pd.DataFrame([], columns = ["x", "y", "w", "h"])

    ###
    for file_name in os.listdir(path1):
        # Verify if file is empty
        if os.path.getsize(path1 + "/" + file_name) == 0:
            print("File " + file_name + " is empty")
            continue

        annot_temp = pd.read_csv(path1 + "/" + file_name, header=None, sep=" ")
        annot_temp.columns = ["label", "x", "y", "w", "h"]
        annot_path1 = pd.concat([annot_path1, annot_temp], ignore_index=True)

    ###
    for file_name in os.listdir(path2):
        # Verify if file is empty
        if os.path.getsize(path2 + "/" + file_name) == 0:
            print("File " + file_name + " is empty")
            continue

        annot_temp = pd.read_csv(path2 + "/" + file_name, header=None, sep=" ")
        annot_temp.columns = ["label", "x", "y", "w", "h"]
        annot_path2 = pd.concat([annot_path2, annot_temp], ignore_index=True)

    for file_name in os.listdir(path3):
        # Verify if file is empty
        if os.path.getsize(path3 + "/" + file_name) == 0:
            print("File " + file_name + " is empty")
            continue

        annot_temp = pd.read_csv(path3 + "/" + file_name, header=None, sep=" ")
        annot_temp.columns = ["label", "x", "y", "w", "h"]
        annot_path3 = pd.concat([annot_path3, annot_temp], ignore_index=True)

    annot = pd.concat([annot_path1, annot_path2, annot_path3], ignore_index=True)

    print("Annot - Number of " + labels[0] + ": " + str(annot_path1.shape[0]))
    print("Annot - Number of " + labels[1] + ": " + str(annot_path2.shape[0]))
    print("Annot - Number of " + labels[2] + ": " + str(annot_path3.shape[0]))

def verify_number_images(path1, path2, path3, labels):
    
    print("Images - Number of " + labels[0] + ": " + str(len([file_name for file_name in os.listdir(path1)])))
    print("Images - Number of " + labels[1] + ": " + str(len([file_name for file_name in os.listdir(path2)])))
    print("Images - Number of " + labels[2] + ": " + str(len([file_name for file_name in os.listdir(path3)])))


#### 1 - Creating Uniclass Datasets without outliers

In [3]:
import os

from PIL import Image
import pandas as pd
import os.path
import shutil
import numpy as np
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

def create_files_without_outliers_uniclass(directory, new_directory, labels, higher_limit):

    # Create Folders

    if not os.path.isdir(new_directory):

        print("Creating Folders....")
        print("==================================")
        print("\n")

        os.mkdir(new_directory)
        print(new_directory)

        sub_folders_1 = ["images", "labels"]
        sub_folders_2 = ["test", "train"]

        if not isinstance(labels, list):
            labels_folders = [labels]
        else:
            labels_folders = labels

        for sub_folder_1 in sub_folders_1:
            os.mkdir(new_directory + "/" + sub_folder_1)
            print(new_directory + "/" + sub_folder_1)

            for sub_folder_2 in sub_folders_2:
                os.mkdir(new_directory + "/" + sub_folder_1 + "/" + sub_folder_2)
                print(new_directory + "/" + sub_folder_1 + "/" + sub_folder_2)

                for labels_folder in labels_folders:
                    os.mkdir(new_directory + "/" + sub_folder_1 + "/" + sub_folder_2 + "/" + labels_folder)
                    print(new_directory + "/" + sub_folder_1 + "/" + sub_folder_2 + "/" + labels_folder)

    shape = []
     
    image_path = directory + "/images/test/" + labels

    print(image_path)

    for filename in os.listdir(image_path):
        file_path = image_path + "/" + filename
        img = Image.open(file_path)
        shape = np.array(img).shape

        print(file_path)
        print(shape)
        
        break

    empty_csv = pd.DataFrame([], columns=["Empty Files", "Label"])
    
    print("==================================")
    print("Image Shape: " + str(shape[0]) + ", " + str(shape[1]))

    annot_old_directory = [
        directory + "/labels/test/" + labels,
        directory + "/labels/train/" + labels
    ]

    annot_new_directory = [
        new_directory + "/labels/test/" + labels,
        new_directory + "/labels/train/" + labels
    ]

    images_old_directory = [
        directory + "/images/test/" + labels,
        directory + "/images/train/" + labels
    ]

    images_new_directory = [
        new_directory + "/images/test/" + labels,
        new_directory + "/images/train/" + labels
    ]

    print("Filtering the annotation....")
    print("==================================")
    print("\n")


    # Iterate through all files in the specified directory
    for direct, new_direct in zip(annot_old_directory, annot_new_directory):
        print("Annot Label: " + labels)
        print("Folder: " + direct)
        print("==================================")

        for filename in os.listdir(direct):
            # Check if the file is a text file
            if filename.endswith('.txt'):  # Adjust the extension as needed
                file_path = os.path.join(direct, filename)
                df = pd.read_csv(file_path, sep=' ', names=["Labels", "x", "y", "w", "h"])
            
                for index, row in df.iterrows():
                    if (float(row["w"]) * shape[0]) * (float(row["h"]) * shape[1]) > higher_limit: 
                        df = df.drop(index=index)
                
                if not df.empty:
                    df.to_csv(new_direct + "/" + filename, sep=" ", index=False, header=False)
                else:
                    empty_csv.loc[-1] = [filename, labels]
                    empty_csv.index += 1
                    empty_csv = empty_csv.sort_index()


    print("Removing the images without annotation after filtering....")
    print("==================================")
    print("\n")
        
    counter = 0
    for direct, new_direct in zip(images_old_directory, images_new_directory):     
        
        print("Images Label: " + labels)
        print("Folder: " + direct)
        print("==================================")
        
        for filename in os.listdir(direct):

            image_extension = filename.split(".")[-1]

            if not empty_csv["Empty Files"].isin([filename.replace(image_extension, "txt")]).any():
                shutil.copyfile(direct + "/" + filename, new_direct + "/" + filename)
            else:
                counter+=1

    print("Removed images for " + new_direct + ": " + str(counter))
    print("==================================")

#### 2 - Uniclass Datasets

##### 2.1 - Constants and path

In [4]:
main_path_all_data = "DatasetOrganized/AllData"
main_path_without_outliers_1_20 = "DatasetOrganized/WithoutOutliers_1_20"
main_path_without_outliers_1_50 = "DatasetOrganized/WithoutOutliers_1_50"

area_higher_fence_1_20_list = [1331.923805, 1517.573538, 871.728119] 
area_higher_fence_1_50_list = [1476.435217, 1682.454364, 965.706874] 
                    
sub_folders_dataset_all_data = ["DatasetBlackRot", "DatasetEsca", "DatasetLeafBlight"]
sub_folders_dataset_without_outliers_1_20 = ["DatasetBlackRot_WithoutOutliers_1_20", "DatasetEsca_WithoutOutliers_1_20", "DatasetLeafBlight_WithoutOutliers_1_20"]
sub_folders_dataset_without_outliers_1_50 = ["DatasetBlackRot_WithoutOutliers_1_50", "DatasetEsca_WithoutOutliers_1_50", "DatasetLeafBlight_WithoutOutliers_1_50"]

labels = ["BlackRot", "Esca", "LeafBlight"]

##### 2.2 - Create Datasets without 2.5% of outliers, for each label

In [5]:
for sub_folder_all_data, sub_folder_without_outliers_1_20, area_higher_fence_1_20, label in zip(sub_folders_dataset_all_data, sub_folders_dataset_without_outliers_1_20, area_higher_fence_1_20_list, labels):

    old_directory = main_path_all_data + "/" + sub_folder_all_data
    new_directory = main_path_without_outliers_1_20 + "/" + sub_folder_without_outliers_1_20

    create_files_without_outliers_uniclass(old_directory, new_directory, label, area_higher_fence_1_20)

DatasetOrganized/AllData/DatasetBlackRot/images/test/BlackRot
DatasetOrganized/AllData/DatasetBlackRot/images/test/BlackRot/00090b0f-c140-4e77-8d20-d39f67b75fcc___FAM_B.Rot 0376.JPG
(256, 256, 3)
Image Shape: 256, 256
Filtering the annotation....


Annot Label: BlackRot
Folder: DatasetOrganized/AllData/DatasetBlackRot/labels/test/BlackRot
Annot Label: BlackRot
Folder: DatasetOrganized/AllData/DatasetBlackRot/labels/train/BlackRot
Removing the images without annotation after filtering....


Images Label: BlackRot
Folder: DatasetOrganized/AllData/DatasetBlackRot/images/test/BlackRot
Images Label: BlackRot
Folder: DatasetOrganized/AllData/DatasetBlackRot/images/train/BlackRot
Removed images for DatasetOrganized/WithoutOutliers_1_20/DatasetBlackRot_WithoutOutliers_1_20/images/train/BlackRot: 23
DatasetOrganized/AllData/DatasetEsca/images/test/Esca
DatasetOrganized/AllData/DatasetEsca/images/test/Esca/0075b632-2e34-4e4f-9697-fe2b332b7ef8___FAM_B.Msls 4399_flipLR.JPG
(256, 256, 3)
Image Shap

##### 2.3 - Create Datasets without 5% of outliers, for each label

In [6]:
for sub_folder_all_data, sub_folder_without_outliers_1_50, area_higher_fence_1_50, label in zip(sub_folders_dataset_all_data, sub_folders_dataset_without_outliers_1_50, area_higher_fence_1_50_list, labels):

    old_directory = main_path_all_data + "/" + sub_folder_all_data
    new_directory = main_path_without_outliers_1_50 + "/" + sub_folder_without_outliers_1_50

    create_files_without_outliers_uniclass(old_directory, new_directory, label, area_higher_fence_1_50)

DatasetOrganized/AllData/DatasetBlackRot/images/test/BlackRot
DatasetOrganized/AllData/DatasetBlackRot/images/test/BlackRot/00090b0f-c140-4e77-8d20-d39f67b75fcc___FAM_B.Rot 0376.JPG
(256, 256, 3)
Image Shape: 256, 256
Filtering the annotation....


Annot Label: BlackRot
Folder: DatasetOrganized/AllData/DatasetBlackRot/labels/test/BlackRot
Annot Label: BlackRot
Folder: DatasetOrganized/AllData/DatasetBlackRot/labels/train/BlackRot
Removing the images without annotation after filtering....


Images Label: BlackRot
Folder: DatasetOrganized/AllData/DatasetBlackRot/images/test/BlackRot
Images Label: BlackRot
Folder: DatasetOrganized/AllData/DatasetBlackRot/images/train/BlackRot
Removed images for DatasetOrganized/WithoutOutliers_1_50/DatasetBlackRot_WithoutOutliers_1_50/images/train/BlackRot: 16
DatasetOrganized/AllData/DatasetEsca/images/test/Esca
DatasetOrganized/AllData/DatasetEsca/images/test/Esca/0075b632-2e34-4e4f-9697-fe2b332b7ef8___FAM_B.Msls 4399_flipLR.JPG
(256, 256, 3)
Image Shap

#### 2.4 - Verify the annot and image for each uniclass dataset

In [7]:

for folder_all_data, folder_1_20, folder_1_50, label in zip(sub_folders_dataset_all_data, sub_folders_dataset_without_outliers_1_20, sub_folders_dataset_without_outliers_1_50, labels):
    for folder_train in ["train", "test"]:
        # With all Outliers
        path_all_data = main_path_all_data + "/" + folder_all_data + "/labels/" + folder_train + "/" + label

        # Without Outliers (1.2)
        path_1_2 = main_path_without_outliers_1_20 + "/" + folder_1_20 + "/labels/" + folder_train + "/" + label

        # Without Outliers (1.5)
        path_1_5 = main_path_without_outliers_1_50 + "/" + folder_1_50 + "/labels/" + folder_train + "/" + label

        verify_number_annot(path_all_data, path_1_2, path_1_5, ["With Outliers", "Without Outliers (1.2)", "Without Outliers (1.5)"])

        print("\n")

        # With all Outliers
        path_all_data = main_path_all_data + "/" + folder_all_data + "/labels/" + folder_train + "/" + label

        # Without Outliers (1.2)
        path_1_2 = main_path_without_outliers_1_20 + "/" + folder_1_20 + "/labels/" + folder_train + "/" + label

        # Without Outliers (1.5)
        path_1_5 = main_path_without_outliers_1_50 + "/" + folder_1_50 + "/labels/" + folder_train + "/" + label

        verify_number_images(path_all_data, path_1_2, path_1_5, ["With Outliers", "Without Outliers (1.2)", "Without Outliers (1.5)"])

        print("\n\n")



Annot - Number of With Outliers: 17685
Annot - Number of Without Outliers (1.2): 16731
Annot - Number of Without Outliers (1.5): 16990


Images - Number of With Outliers: 2168
Images - Number of Without Outliers (1.2): 2149
Images - Number of Without Outliers (1.5): 2154



Annot - Number of With Outliers: 3918
Annot - Number of Without Outliers (1.2): 3424
Annot - Number of Without Outliers (1.5): 3567


Images - Number of With Outliers: 508
Images - Number of Without Outliers (1.2): 504
Images - Number of Without Outliers (1.5): 506



Annot - Number of With Outliers: 24491
Annot - Number of Without Outliers (1.2): 22266
Annot - Number of Without Outliers (1.5): 22723


Images - Number of With Outliers: 1919
Images - Number of Without Outliers (1.2): 1917
Images - Number of Without Outliers (1.5): 1918



Annot - Number of With Outliers: 5098
Annot - Number of Without Outliers (1.2): 4662
Annot - Number of Without Outliers (1.5): 4753


Images - Number of With Outliers: 480
Images - 

#### 3 - Creating MultiClass Datasets without outliers

In [8]:
import pandas as pd
import os.path

def create_files_without_outliers_multiclass(directory, new_directory, labels, higher_limits):


    if not os.path.isdir(new_directory):

        print("Creating Folders....")
        print("==================================")
        print("\n")

        os.mkdir(new_directory)
        print(new_directory)

        sub_folders_1 = ["images", "labels"]
        sub_folders_2 = ["test", "train"]

        if not isinstance(labels, list):
            labels_folders = [labels]
        else:
            labels_folders = labels

        for sub_folder_1 in sub_folders_1:
            os.mkdir(new_directory + "/" + sub_folder_1)
            print(new_directory + "/" + sub_folder_1)

            for sub_folder_2 in sub_folders_2:
                os.mkdir(new_directory + "/" + sub_folder_1 + "/" + sub_folder_2)
                print(new_directory + "/" + sub_folder_1 + "/" + sub_folder_2)

                for labels_folder in labels_folders:
                    os.mkdir(new_directory + "/" + sub_folder_1 + "/" + sub_folder_2 + "/" + labels_folder)
                    print(new_directory + "/" + sub_folder_1 + "/" + sub_folder_2 + "/" + labels_folder)


    

    for label, higher_limit in zip(labels, higher_limits):

        shape = []
        
        image_path = directory + "/images/test/" + label

        empty_csv = pd.DataFrame([], columns=["Empty Files", "Label"])

        for filename in os.listdir(image_path):
            file_path = os.path.join(image_path, filename)
            img = Image.open(file_path)
            shape = np.array(img).shape
            
            break

            
                

        
        
        print("==================================")
        print("Image Shape: " + str(shape[0]) + ", " + str(shape[0]))

        annot_old_directory = []
        annot_new_directory = []
    

        annot_old_directory.append(directory + "/labels/test/" + label)
        annot_old_directory.append(directory + "/labels/train/" + label)    

        annot_new_directory.append(new_directory + "/labels/test/" + label)
        annot_new_directory.append(new_directory + "/labels/train/" + label)

        images_old_directory = [
            directory + "/images/test/" + label,
            directory + "/images/train/" + label
        ]

        images_new_directory = [
            new_directory + "/images/test/" + label,
            new_directory + "/images/train/" + label
        ]
    
        print("\n\n\n\n")

        print("Filtering the annotation....")
        print("==================================")
        print("\n")


        # Iterate through all files in the specified directory
        for direct, new_direct in zip(annot_old_directory, annot_new_directory):
            print("Label: " + label)
            print("Folder: " + direct)
            print("==================================")

            for filename in os.listdir(direct):
                # Check if the file is a text file
                if filename.endswith('.txt'):  # Adjust the extension as needed
                    file_path = os.path.join(direct, filename)
                    df = pd.read_csv(file_path, sep=' ', names=["Labels", "x", "y", "w", "h"])
                
                    for index, row in df.iterrows():
                        if (float(row["w"]) * shape[0]) * (float(row["h"]) * shape[1]) > higher_limit: 
                            df = df.drop(index=index)
                    
                    if not df.empty:
                        df.to_csv(new_direct + "/" + filename, sep=" ", index=False, header=False)
                    else:
                        empty_csv.loc[-1] = [filename, labels]
                        empty_csv.index += 1
                        empty_csv = empty_csv.sort_index()
                        

        print("Removing the images without annotation after filtering....")
        print("==================================")
        print("\n")
            
        counter = 0
        for direct, new_direct in zip(images_old_directory, images_new_directory):     
            
            print("Images Label: " + label)
            print("Folder: " + direct)
            print("==================================")
            
            for filename in os.listdir(direct):

                image_extension = filename.split(".")[-1]

                if not empty_csv["Empty Files"].isin([filename.replace(image_extension, "txt")]).any():
                    shutil.copyfile(direct + "/" + filename, new_direct + "/" + filename)
                else:
                    counter+=1

        print("Removed images for " + new_direct + ": " + str(counter))
        print("==================================")
        

#### 4. Multiclass Dataset

##### 4.1 - Constants and path

In [9]:
main_path_all_data = "DatasetOrganized/AllData"
main_path_without_outliers_1_20 = "DatasetOrganized/WithoutOutliers_1_20"
main_path_without_outliers_1_50 = "DatasetOrganized/WithoutOutliers_1_50"

area_higher_fence_1_20_list = [1331.923805, 1517.573538, 871.728119] 
area_higher_fence_1_50_list = [1476.435217, 1682.454364, 965.706874] 
                    
sub_folders_dataset_all_data = "Dataset3Classes"
sub_folders_dataset_without_outliers_1_20 = "Dataset3Classes_WithoutOutliers_1_20"
sub_folders_dataset_without_outliers_1_50 = "Dataset3Classes_WithoutOutliers_1_50"

labels = ["BlackRot", "Esca", "LeafBlight"]

##### 4.2 - Create Dataset without 2.5% of outliers, for each label

In [10]:
old_directory = main_path_all_data + "/" + sub_folders_dataset_all_data
new_directory = main_path_without_outliers_1_20 + "/" + sub_folders_dataset_without_outliers_1_20

create_files_without_outliers_multiclass(old_directory, new_directory, labels, area_higher_fence_1_20_list)

Image Shape: 256, 256





Filtering the annotation....


Label: BlackRot
Folder: DatasetOrganized/AllData/Dataset3Classes/labels/test/BlackRot
Label: BlackRot
Folder: DatasetOrganized/AllData/Dataset3Classes/labels/train/BlackRot
Removing the images without annotation after filtering....


Images Label: BlackRot
Folder: DatasetOrganized/AllData/Dataset3Classes/images/test/BlackRot
Images Label: BlackRot
Folder: DatasetOrganized/AllData/Dataset3Classes/images/train/BlackRot
Removed images for DatasetOrganized/WithoutOutliers_1_20/Dataset3Classes_WithoutOutliers_1_20/images/train/BlackRot: 23
Image Shape: 256, 256





Filtering the annotation....


Label: Esca
Folder: DatasetOrganized/AllData/Dataset3Classes/labels/test/Esca
Label: Esca
Folder: DatasetOrganized/AllData/Dataset3Classes/labels/train/Esca
Removing the images without annotation after filtering....


Images Label: Esca
Folder: DatasetOrganized/AllData/Dataset3Classes/images/test/Esca
Images Label: Esca
Folder: DatasetOrgani

##### 4.3 - Create Dataset without 5% of outliers, for each label

In [11]:
old_directory = main_path_all_data + "/" + sub_folders_dataset_all_data
new_directory = main_path_without_outliers_1_50 + "/" + sub_folders_dataset_without_outliers_1_50

create_files_without_outliers_multiclass(old_directory, new_directory, labels, area_higher_fence_1_50_list)

Image Shape: 256, 256





Filtering the annotation....


Label: BlackRot
Folder: DatasetOrganized/AllData/Dataset3Classes/labels/test/BlackRot
Label: BlackRot
Folder: DatasetOrganized/AllData/Dataset3Classes/labels/train/BlackRot
Removing the images without annotation after filtering....


Images Label: BlackRot
Folder: DatasetOrganized/AllData/Dataset3Classes/images/test/BlackRot
Images Label: BlackRot
Folder: DatasetOrganized/AllData/Dataset3Classes/images/train/BlackRot
Removed images for DatasetOrganized/WithoutOutliers_1_50/Dataset3Classes_WithoutOutliers_1_50/images/train/BlackRot: 16
Image Shape: 256, 256





Filtering the annotation....


Label: Esca
Folder: DatasetOrganized/AllData/Dataset3Classes/labels/test/Esca
Label: Esca
Folder: DatasetOrganized/AllData/Dataset3Classes/labels/train/Esca
Removing the images without annotation after filtering....


Images Label: Esca
Folder: DatasetOrganized/AllData/Dataset3Classes/images/test/Esca
Images Label: Esca
Folder: DatasetOrgani

In [ ]:
for folder_all_data, folder_1_20, folder_1_50, label in zip([sub_folders_dataset_all_data], [sub_folders_dataset_without_outliers_1_20], [sub_folders_dataset_without_outliers_1_50], labels):
    for folder_train in ["train", "test"]:
        # With all Outliers

        ### TENS DE FAZER DE NOVO

        path_all_data = main_path_all_data + "/" + folder_all_data + "/labels/" + folder_train + "/" + label

        # Without Outliers (1.2)
        path_1_2 = main_path_without_outliers_1_20 + "/" + folder_1_20 + "/labels/" + folder_train + "/" + label

        # Without Outliers (1.5)
        path_1_5 = main_path_without_outliers_1_50 + "/" + folder_1_50 + "/labels/" + folder_train + "/" + label

        verify_number_annot(path_all_data, path_1_2, path_1_5, ["With Outliers", "Without Outliers (1.2)", "Without Outliers (1.5)"])

        print("\n")

        # With all Outliers
        path_all_data = main_path_all_data + "/" + folder_all_data + "/labels/" + folder_train + "/" + label

        # Without Outliers (1.2)
        path_1_2 = main_path_without_outliers_1_20 + "/" + folder_1_20 + "/labels/" + folder_train + "/" + label

        # Without Outliers (1.5)
        path_1_5 = main_path_without_outliers_1_50 + "/" + folder_1_50 + "/labels/" + folder_train + "/" + label

        verify_number_images(path_all_data, path_1_2, path_1_5, ["With Outliers", "Without Outliers (1.2)", "Without Outliers (1.5)"])

        print("\n\n")



Annot - Number of With Outliers: 17685
Annot - Number of Without Outliers (1.2): 16731
Annot - Number of Without Outliers (1.5): 16990


Images - Number of With Outliers: 2168
Images - Number of Without Outliers (1.2): 2149
Images - Number of Without Outliers (1.5): 2154



Annot - Number of With Outliers: 3918
Annot - Number of Without Outliers (1.2): 3424
Annot - Number of Without Outliers (1.5): 3567


Images - Number of With Outliers: 508
Images - Number of Without Outliers (1.2): 504
Images - Number of Without Outliers (1.5): 506





In [12]:
import cv2
import matplotlib.pyplot as plt
import os
from PIL import Image
import numpy as np
import pandas as pd


main_path = "Dataset/AllData/Dataset3Classes"

black_rot_path_img_train = main_path + "/images/train/BlackRot"
black_rot_path_annot_train = main_path + "/labels/train/BlackRot"

black_rot_path_img_test = main_path + "/images/test/BlackRot"
black_rot_path_annot_test = main_path + "/labels/test/BlackRot"

esca_path_img_train = main_path + "/images/train/Esca"
esca_path_annot_train = main_path + "/labels/train/Esca"

esca_path_img_test = main_path + "/images/test/Esca"
esca_path_annot_test = main_path + "/labels/test/Esca"

leaf_blight_path_img_train = main_path + "/images/train/LeafBlight"
leaf_blight_path_annot_train = main_path + "/labels/train/LeafBlight"

leaf_blight_path_img_test = main_path + "/images/test/LeafBlight"
leaf_blight_path_annot_test = main_path + "/labels/test/LeafBlight"


annot_br = pd.DataFrame([], columns = ["label", "x", "y", "w", "h", "file_name"])
annot_esca = pd.DataFrame([], columns = ["label", "x", "y", "w", "h", "file_name"])
annot_leafbl = pd.DataFrame([], columns = ["label", "x", "y", "w", "h", "file_name"])


###
for file_name in os.listdir(black_rot_path_annot_train):
    annot_temp = pd.read_csv(black_rot_path_annot_train + "/" + file_name, header=None, sep=" ")
    annot_temp.columns = ["label", "x", "y", "w", "h"]
    annot_temp["file_name"] = file_name
    annot_br = pd.concat([annot_br, annot_temp], ignore_index=True)

for file_name in os.listdir(black_rot_path_annot_test):
    annot_temp = pd.read_csv(black_rot_path_annot_test + "/" + file_name, header=None, sep=" ")
    annot_temp.columns = ["label", "x", "y", "w", "h"]
    annot_temp["file_name"] = file_name
    annot_br = pd.concat([annot_br, annot_temp], ignore_index=True)

for file_name in os.listdir(esca_path_annot_train):
    annot_temp = pd.read_csv(esca_path_annot_train + "/" + file_name, header=None, sep=" ")
    annot_temp.columns = ["label", "x", "y", "w", "h"]
    annot_temp["file_name"] = file_name
    annot_esca = pd.concat([annot_esca, annot_temp], ignore_index=True)

for file_name in os.listdir(esca_path_annot_test):
    annot_temp = pd.read_csv(esca_path_annot_test + "/" + file_name, header=None, sep=" ")
    annot_temp.columns = ["label", "x", "y", "w", "h"]
    annot_temp["file_name"] = file_name
    annot_esca = pd.concat([annot_esca, annot_temp], ignore_index=True)

for file_name in os.listdir(leaf_blight_path_annot_train):
    annot_temp = pd.read_csv(leaf_blight_path_annot_train + "/" + file_name, header=None, sep=" ")
    annot_temp.columns = ["label", "x", "y", "w", "h"]
    annot_temp["file_name"] = file_name
    annot_leafbl = pd.concat([annot_leafbl, annot_temp], ignore_index=True)

for file_name in os.listdir(leaf_blight_path_annot_test):
    annot_temp = pd.read_csv(leaf_blight_path_annot_test + "/" + file_name, header=None, sep=" ")
    annot_temp.columns = ["label", "x", "y", "w", "h"]
    annot_temp["file_name"] = file_name
    annot_leafbl = pd.concat([annot_leafbl, annot_temp], ignore_index=True)


###
annot_br["w"] = annot_br["w"].astype(float)
annot_br["h"] = annot_br["h"].astype(float)
annot_br["x"] = (annot_br["x"] - annot_br["w"]/2).astype(float)
annot_br["y"] = (annot_br["y"] - annot_br["h"]/2).astype(float)

annot_esca["w"] = annot_esca["w"].astype(float)
annot_esca["h"] = annot_esca["h"].astype(float)
annot_esca["x"] = (annot_esca["x"] - annot_esca["w"]/2).astype(float)
annot_esca["y"] = (annot_esca["y"] - annot_esca["h"]/2).astype(float)


annot_leafbl["w"] = annot_leafbl["w"].astype(float)
annot_leafbl["h"] = annot_leafbl["h"].astype(float)
annot_leafbl["x"] = (annot_leafbl["x"] - annot_leafbl["w"]/2).astype(float)
annot_leafbl["y"] = (annot_leafbl["y"] - annot_leafbl["h"]/2).astype(float)

annot = pd.concat([annot_br, annot_esca, annot_leafbl], ignore_index=True)

In [13]:
print("BlackRot Annot: " + str(annot[annot["label"] == 0].shape))
print("Esca Annot: " + str(annot[annot["label"] == 1].shape))
print("LeafBlight Annot: " + str(annot[annot["label"] == 2].shape))

BlackRot Annot: (21603, 6)
Esca Annot: (29589, 6)
LeafBlight Annot: (33524, 6)
